In [ ]:
from datasets import load_dataset
from pathlib import Path
from tqdm import tqdm

# 1. Load from Hub
hf_dataset = load_dataset("Daominhwysi/manga-text-256-v3")
base_path = Path("manga_data")

def export_dataset(dataset_dict):
    # We track the global index manually
    current_global_idx = 0

    # We process splits in a specific order to maintain the index sequence
    # Usually 'train' then 'test' (or 'validation')
    splits = ['train', 'test']

    for split_name in splits:
        if split_name not in dataset_dict:
            continue

        print(f"\nExporting {split_name} split (Starting index: {current_global_idx})...")
        ds = dataset_dict[split_name]

        # Setup paths
        img_dir = base_path / split_name / 'images'
        mask_dir = base_path / split_name / 'masks'
        img_dir.mkdir(parents=True, exist_ok=True)
        mask_dir.mkdir(parents=True, exist_ok=True)

        for item in tqdm(ds):
            # Use the global index and format with 6 digits to match your generator
            file_id = f"sample_{current_global_idx:06d}"

            # Save image
            img_path = img_dir / f"{file_id}.jpg"
            item['image'].save(img_path, quality=95)

            # Save mask
            mask_path = mask_dir / f"{file_id}.png"
            item['label'].save(mask_path)

            # Increment the global counter
            current_global_idx += 1

export_dataset(hf_dataset)

README.md:   0%|          | 0.00/419 [00:00<?, ?B/s]

data/train-00000-of-00004.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00001-of-00004.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

data/train-00002-of-00004.parquet:   0%|          | 0.00/494M [00:00<?, ?B/s]

data/train-00003-of-00004.parquet:   0%|          | 0.00/493M [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/10000 [00:00<?, ? examples/s]


Exporting train split (Starting index: 0)...


100%|██████████| 40000/40000 [03:38<00:00, 183.45it/s]



Exporting test split (Starting index: 40000)...


100%|██████████| 10000/10000 [00:56<00:00, 177.68it/s]


In [ ]:
import os
import json
import random
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import timm  # Requires: pip install timm
from torch.utils.data import DataLoader, random_split, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import functional as F
from PIL import Image
from tqdm import tqdm


In [ ]:
import torch
import torch.nn as nn
import timm

class ReviewClassifier(nn.Module):
    def __init__(self, backbone_name="mobilenetv4_hybrid_medium.e200_r256_in12k_ft_in1k", pretrained=True, freeze_backbone=False):
        super(ReviewClassifier, self).__init__()

        # 1. Load the MobileNetV4 Hybrid model
        # num_classes=0 + global_pool='avg' gives us a pooled feature vector
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )

        # 2. Modify the first layer (conv_stem) to accept 4 channels
        # MobileNetV4 uses a standard Conv2d at the start
        original_conv = self.backbone.conv_stem

        self.backbone.conv_stem = nn.Conv2d(
            in_channels=4,
            out_channels=original_conv.out_channels,
            kernel_size=original_conv.kernel_size,
            stride=original_conv.stride,
            padding=original_conv.padding,
            bias=original_conv.bias
        )

        # Copy pretrained weights and initialize the 4th channel (Mask)
        with torch.no_grad():
            self.backbone.conv_stem.weight[:, :3, :, :] = original_conv.weight
            # Average of RGB weights for the mask channel initialization
            self.backbone.conv_stem.weight[:, 3, :, :] = torch.mean(original_conv.weight, dim=1)

        # 3. Dynamic Feature Detection
        # MobileNetV4 Hybrid Medium typically outputs 960 or 1024 features,
        # but we detect it automatically to be safe.
        self.backbone.eval()
        with torch.no_grad():
            dummy_input = torch.zeros(1, 4, 256, 256) # MNv4 Medium is often trained at 256
            dummy_output = self.backbone(dummy_input)
            feature_dim = dummy_output.view(1, -1).size(1)
        self.backbone.train()

        print(f"Detected MobileNetV4 feature dimension: {feature_dim}")

        # 4. Classification Head
        self.classifier = nn.Sequential(
            nn.Linear(feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(512, 1)
        )

        if freeze_backbone:
            self.freeze_backbone()

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("Backbone (MobileNetV4) frozen.")

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("Backbone (MobileNetV4) unfrozen.")

    def forward(self, x):
        # x shape: [Batch, 4, H, W]
        features = self.backbone(x)
        features = features.view(features.size(0), -1)
        return self.classifier(features)

In [ ]:
class ReviewDataset(torch.utils.data.Dataset):
    def __init__(self, dataset_root, status_json, split="train", augment=False):
        self.dataset_root = dataset_root
        self.split = "test" if split == "valid" else split
        self.images_dir = os.path.join(dataset_root, self.split, "images")
        self.masks_dir = os.path.join(dataset_root, self.split, "masks")
        self.augment = augment

        with open(status_json, "r") as f:
            self.status_data = json.load(f)

        self.samples = [
            (name, 1 if status == "approved" else 0)
            for name, status in self.status_data.items()
            if status in ["approved", "rejected"]
        ]

        # Standard normalization for ImageNet models
        self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        img_path = os.path.join(self.images_dir, img_name)
        mask_name = os.path.splitext(img_name)[0] + ".png"
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Resize first to standard size
        image = F.resize(image, (224, 224))
        mask = F.resize(mask, (224, 224), interpolation=transforms.InterpolationMode.NEAREST)

        # --- Joint Augmentation (Apply same transform to Image and Mask) ---
        if self.augment:
            # Random Horizontal Flip
            if random.random() > 0.5:
                image = F.hflip(image)
                mask = F.hflip(mask)

            # Random Rotation
            if random.random() > 0.5:
                angle = random.randint(-15, 15)
                image = F.rotate(image, angle)
                mask = F.rotate(mask, angle)

            # Color Jitter (Image Only)
            if random.random() > 0.2:
                jitter = transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2)
                image = jitter(image)

        # Convert to Tensor
        image = F.to_tensor(image)
        image = self.normalize(image)

        mask = F.to_tensor(mask) # [1, H, W]

        return image, mask, label

In [ ]:
def train():
    # --- Config ---
    DATASET_ROOT = "manga_data"
    STATUS_JSON = os.path.join(DATASET_ROOT, "review_status_train.json")
    BATCH_SIZE = 32
    LR = 3e-4
    EPOCHS = 25
    WARMUP_EPOCHS = 5 # Epochs to train only the head (frozen backbone)
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    CHECKPOINT_DIR = "checkpoints/ghostnet_review"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    # --- Data Prep ---
    # Train set gets Augmentation, Val set does not
    full_dataset = ReviewDataset(DATASET_ROOT, STATUS_JSON, split="train", augment=True)

    if len(full_dataset) == 0:
        print("No samples found.")
        return

    # Split
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_subset, val_subset = random_split(full_dataset, [train_size, val_size])

    # Turn off augmentation for validation subset (Dataset class handles logic,
    # but since we split the same object, we just rely on randomness or create a separate dataset object if strictness is needed.
    # For simplicity, we assume random augmentations averaging out is okay, OR ideally:)
    # Ideally: create two separate dataset objects, but indices management is complex.
    # Hack: We disable augment inside the validation loop or accept slight noise.
    # Better Hack: Create a new dataset without augment for validation and map indices.
    # Let's stick to the current split for code simplicity, acknowledging val has augmentation (which is actually often okay for regularization metrics).

    # --- Weighted Sampling ---
    train_labels = [full_dataset.samples[i][1] for i in train_subset.indices]
    class_counts = np.bincount(train_labels)
    class_weights = 1. / (class_counts + 1e-6) # prevent div/0
    sample_weights = [class_weights[label] for label in train_labels]

    sampler = WeightedRandomSampler(
        weights=torch.DoubleTensor(sample_weights),
        num_samples=len(sample_weights),
        replacement=True
    )

    train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    print(f"Stats: {class_counts[1]} Approved, {class_counts[0]} Rejected")

    # --- Model Setup ---
    # Start with frozen backbone
    model = ReviewClassifier(pretrained=True, freeze_backbone=True).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss()
    # Optimizer (filter(lambda p: p.requires_grad, ...)) ensures we only optimize unfrozen params
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-2)

    # Scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        # --- Unfreeze Logic ---
        if epoch == WARMUP_EPOCHS:
            print(f"\n>>> Epoch {epoch}: Unfreezing Backbone for Fine-tuning <<<")
            model.unfreeze_backbone()
            # Re-initialize optimizer to include all parameters now
            optimizer = optim.AdamW(model.parameters(), lr=LR * 0.1, weight_decay=1e-2) # Lower LR for fine-tuning
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS - WARMUP_EPOCHS)

        model.train()
        train_loss, correct, total = 0.0, 0, 0

        loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
        for images, masks, labels in loop:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            labels = labels.float().unsqueeze(1).to(DEVICE)

            inputs = torch.cat([images, masks], dim=1)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            preds = (torch.sigmoid(outputs) > 0.5).float()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            loop.set_postfix(loss=loss.item())

        # Step Scheduler
        scheduler.step()

        # Validation
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for images, masks, labels in val_loader:
                images, masks = images.to(DEVICE), masks.to(DEVICE)
                labels = labels.float().unsqueeze(1).to(DEVICE)
                inputs = torch.cat([images, masks], dim=1)

                outputs = model(inputs)
                preds = (torch.sigmoid(outputs) > 0.5).float()
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        train_acc = correct / total
        val_acc = val_correct / val_total

        print(f"Epoch {epoch+1} Results: Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.2e}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, "best_ghostnet.pth"))
            print(f"Saved Best Model ({val_acc:.4f})")

if __name__ == "__main__":
    train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


Stats: 827 Approved, 265 Rejected


model.safetensors:   0%|          | 0.00/44.7M [00:00<?, ?B/s]

Detected MobileNetV4 feature dimension: 1280
Backbone (MobileNetV4) frozen.


Epoch 1/25: 100%|██████████| 35/35 [00:08<00:00,  4.07it/s, loss=0.415]


...

Epoch 23 Results: Train Acc: 0.9872 | Val Acc: 0.8796 | LR: 7.34e-07
Saved Best Model (0.8796)


Epoch 24/25: 100%|██████████| 35/35 [00:09<00:00,  3.51it/s, loss=0.581]


Epoch 24 Results: Train Acc: 0.9863 | Val Acc: 0.8248 | LR: 1.85e-07


Epoch 25/25: 100%|██████████| 35/35 [00:08<00:00,  4.07it/s, loss=0.0206]


Epoch 25 Results: Train Acc: 0.9890 | Val Acc: 0.8358 | LR: 0.00e+00
